# Model Validation and Forecasting Protocol Audit


> **AUTHORITATIVE ARTIFACT-ONLY AUDIT — NO MODEL TRAINING**  
> Persistence-Enhanced LSTM generation now lives in `03_Deep_Learning_LSTM.ipynb`; Corrected Transformer generation now lives in `04_Transformers.ipynb`. This notebook audits the saved artifacts.

This audit tests split integrity, sequence/target alignment, scaling assumptions, forecast schemas, leakage controls, Protocol A versus Protocol B eligibility, and failure diagnostics. It does not import TensorFlow, construct models, fit models, load checkpoints, regenerate forecasts, or overwrite result artifacts.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.6f}".format)
ARTIFACT_ONLY = True
MODEL_TRAINING_EXECUTED = False
MODEL_CHECKPOINT_LOADED = False
FORECAST_REGENERATED = False


## 1. Dataset and Split Verification


In [2]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
raw_df = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(raw_df)
target = df_daily["Close"].asfreq("D").dropna().rename("Close")
split_idx = int(len(target) * 0.8)
train = target.iloc[:split_idx]
test = target.iloc[split_idx:]
y_test = test.copy()

split_report = pd.DataFrame({
    "Item": ["Training end", "Test start", "Training length", "Test length", "Train/test overlap"],
    "Value": [train.index.max(), test.index.min(), len(train), len(test), len(train.index.intersection(test.index)) > 0],
})
assert train.index.max() < test.index.min()
assert len(test) == 1061
split_report


,Item,Value
0,Training end,2023-08-11 00:00:00+00:00
1,Test start,2023-08-12 00:00:00+00:00
2,Training length,4241
3,Test length,1061
4,Train/test overlap,False


## 2. Forecast Protocol Definitions and Eligibility


In [3]:
protocol_definition = pd.DataFrame(
    {
        "Protocol": ["A", "B"],
        "Name": ["Rolling one-step-ahead", "Recursive multi-step"],
        "Information available at each test date": ["Observed actual values through the previous day", "Final training window plus prior model predictions only"],
        "Actual test feedback": ["Allowed only after each date is observed", "Not allowed"],
        "Current frozen PE-LSTM artifact": ["Eligible", "Not available"],
        "Corrected Transformer artifact": ["No frozen artifact", "No frozen artifact"],
    }
).set_index("Protocol")
protocol_definition


,Name,Information available at each test date,Actual test feedback,Current frozen PE-LSTM artifact,Corrected Transformer artifact
Protocol,,,,,
A,Rolling one-step-ahead,Observed actual values through the previous day,Allowed only after each date is observed,Eligible,No frozen artifact
B,Recursive multi-step,Final training window plus prior model predict...,Not allowed,Not available,No frozen artifact


## 3. Sequence and Target Alignment Audit

The generation notebooks use an input window ending on day `t-1` to predict day `t`. This audit reconstructs the date map without fitting a model.


In [4]:
LOOKBACK_LSTM = 30
sequence_rows = []
combined_index = train.index[-LOOKBACK_LSTM:].append(test.index)
for i in range(LOOKBACK_LSTM, len(combined_index)):
    sequence_rows.append({"input_window_start": combined_index[i - LOOKBACK_LSTM], "input_window_end": combined_index[i - 1], "target_date": combined_index[i]})
test_sequence_map = pd.DataFrame(sequence_rows)
test_sequence_map["next_day_target"] = test_sequence_map["target_date"] == test_sequence_map["input_window_end"] + pd.Timedelta(days=1)
assert pd.DatetimeIndex(test_sequence_map["target_date"]).equals(test.index)
assert test_sequence_map["next_day_target"].all()
test_sequence_map.head(10)


,input_window_start,input_window_end,target_date,next_day_target
0,2023-07-13 00:00:00+00:00,2023-08-11 00:00:00+00:00,2023-08-12 00:00:00+00:00,True
1,2023-07-14 00:00:00+00:00,2023-08-12 00:00:00+00:00,2023-08-13 00:00:00+00:00,True
2,2023-07-15 00:00:00+00:00,2023-08-13 00:00:00+00:00,2023-08-14 00:00:00+00:00,True
3,2023-07-16 00:00:00+00:00,2023-08-14 00:00:00+00:00,2023-08-15 00:00:00+00:00,True
4,2023-07-17 00:00:00+00:00,2023-08-15 00:00:00+00:00,2023-08-16 00:00:00+00:00,True
5,2023-07-18 00:00:00+00:00,2023-08-16 00:00:00+00:00,2023-08-17 00:00:00+00:00,True
6,2023-07-19 00:00:00+00:00,2023-08-17 00:00:00+00:00,2023-08-18 00:00:00+00:00,True
7,2023-07-20 00:00:00+00:00,2023-08-18 00:00:00+00:00,2023-08-19 00:00:00+00:00,True
8,2023-07-21 00:00:00+00:00,2023-08-19 00:00:00+00:00,2023-08-20 00:00:00+00:00,True
9,2023-07-22 00:00:00+00:00,2023-08-20 00:00:00+00:00,2023-08-21 00:00:00+00:00,True


## 4. Scaling and Inverse-Scaling Audit


In [5]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_values = train.to_numpy().reshape(-1, 1)
test_values = test.to_numpy().reshape(-1, 1)
scaler.fit(train_values)
train_scaled = scaler.transform(train_values)
test_scaled = scaler.transform(test_values)
roundtrip_train = scaler.inverse_transform(train_scaled).ravel()
roundtrip_test = scaler.inverse_transform(test_scaled).ravel()

scaling_audit = pd.DataFrame({
    "Check": ["Scaler fitted only on training data", "Train inverse-transform roundtrip", "Test inverse-transform roundtrip", "Test exceeds training scale range"],
    "Result": [True, np.allclose(roundtrip_train, train_values.ravel()), np.allclose(roundtrip_test, test_values.ravel()), bool((test_scaled < 0).any() or (test_scaled > 1).any())],
})
scaling_audit


,Check,Result
0,Scaler fitted only on training data,True
1,Train inverse-transform roundtrip,True
2,Test inverse-transform roundtrip,True
3,Test exceeds training scale range,True


## 5. Deterministic Baseline Audit


In [6]:
naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
moving_average_forecast = target.shift(1).rolling(window=7).mean().reindex(test.index).rename("7-Day Moving Average")
baseline_audit = pd.DataFrame({
    "Check": ["Naive equals prior observed close", "Moving average uses seven prior values", "No missing baseline forecasts"],
    "Pass": [naive_forecast.equals(target.shift(1).reindex(test.index)), all(target.loc[:date].iloc[:-1].tail(7).index.max() < date for date in test.index), not pd.concat([naive_forecast, moving_average_forecast], axis=1).isna().any().any()],
})
assert baseline_audit["Pass"].all()
baseline_audit


,Check,Pass
0,Naive equals prior observed close,True
1,Moving average uses seven prior values,True
2,No missing baseline forecasts,True


## 6. Saved Forecast Artifact Inventory

The PE-LSTM has a frozen authoritative vector. The corrected Transformer remains an exploratory failure-case model and has no frozen result CSV; its deterministic generation and structural checks now live in Notebook 04.


In [7]:
results_dir = PROJECT_ROOT / "results"
persistence_path = results_dir / "persistence_enhanced_lstm_forecast.csv"
transformer_candidates = sorted(results_dir.glob("*transformer*forecast*.csv"))
artifact_inventory = pd.DataFrame([
    {"Model": "Persistence-Enhanced LSTM", "Expected path": str(persistence_path), "Exists": persistence_path.exists(), "Audit treatment": "Full saved-vector audit"},
    {"Model": "Corrected Transformer", "Expected path": "No authoritative result artifact", "Exists": bool(transformer_candidates), "Audit treatment": "Generation/structure audited in Notebook 04; excluded from frozen ranking"},
]).set_index("Model")
assert persistence_path.exists()
artifact_inventory


,Expected path,Exists,Audit treatment
Model,,,
Persistence-Enhanced LSTM,E:\Re_Sc_AM\TimeSeriesFoundationModels\results...,True,Full saved-vector audit
Corrected Transformer,No authoritative result artifact,False,Generation/structure audited in Notebook 04; e...


## 7. Persistence-Enhanced LSTM Artifact Audit


In [8]:
persistence_artifact = pd.read_csv(persistence_path, parse_dates=["Timestamp"])
persistence_artifact["Timestamp"] = pd.to_datetime(persistence_artifact["Timestamp"], utc=True)
persistence_forecast = persistence_artifact.set_index("Timestamp")["Persistence_Enhanced_LSTM"].rename("Persistence-Enhanced LSTM")

persistence_checks = pd.DataFrame({
    "Check": ["Shape is 1061 x 2", "Expected schema", "Exact test timestamps", "Unique sorted timestamps", "No missing values", "Finite forecasts", "No model training executed"],
    "Pass": [persistence_artifact.shape == (1061, 2), persistence_artifact.columns.tolist() == ["Timestamp", "Persistence_Enhanced_LSTM"], persistence_forecast.index.equals(test.index), persistence_forecast.index.is_unique and persistence_forecast.index.is_monotonic_increasing, not persistence_artifact.isna().any().any(), np.isfinite(persistence_forecast).all(), not MODEL_TRAINING_EXECUTED],
})
assert persistence_checks["Pass"].all()
persistence_metrics = pd.DataFrame([{"Model": "Persistence-Enhanced LSTM", "MAE": mae(y_test, persistence_forecast), "RMSE": rmse(y_test, persistence_forecast), "MAPE": mape(y_test, persistence_forecast), "sMAPE": smape(y_test, persistence_forecast)}]).set_index("Model")
display(persistence_checks)
persistence_metrics


,Check,Pass
0,Shape is 1061 x 2,True
1,Expected schema,True
2,Exact test timestamps,True
3,Unique sorted timestamps,True
4,No missing values,True
5,Finite forecasts,True
6,No model training executed,True


,MAE,RMSE,MAPE,sMAPE
Model,,,,
Persistence-Enhanced LSTM,1321.365311,1881.091190,1.783956,1.791645


## 8. Forecast Diagnostics and Protocol-A Comparison


In [9]:
def prediction_diagnostics(name, forecast, actual):
    aligned = pd.concat([actual.rename("actual"), forecast.rename("prediction")], axis=1).dropna()
    return pd.Series({"Model": name, "N": len(aligned), "Prediction unique values": aligned["prediction"].nunique(), "Prediction std": aligned["prediction"].std(), "Actual std": aligned["actual"].std(), "Prediction change std": aligned["prediction"].diff().std(), "Actual change std": aligned["actual"].diff().std(), "Prediction min": aligned["prediction"].min(), "Prediction max": aligned["prediction"].max()})

protocol_a_forecasts = {"Naive": naive_forecast, "7-Day Moving Average": moving_average_forecast, "Persistence-Enhanced LSTM": persistence_forecast}
protocol_a_metrics = pd.DataFrame([{"Model": name, "MAE": mae(y_test, forecast), "RMSE": rmse(y_test, forecast), "MAPE": mape(y_test, forecast), "sMAPE": smape(y_test, forecast)} for name, forecast in protocol_a_forecasts.items()]).set_index("Model").sort_values("RMSE")
forecast_diagnostics = pd.DataFrame([prediction_diagnostics(name, forecast, y_test) for name, forecast in protocol_a_forecasts.items()]).set_index("Model")
display(forecast_diagnostics)
protocol_a_metrics


,N,Prediction unique values,Prediction std,Actual std,Prediction change std,Actual change std,Prediction min,Prediction max
Model,,,,,,,,
Naive,1061,1053,25600.549440,25564.279690,1855.084432,1855.088666,25155.000000,124728.000000
7-Day Moving Average,1061,1061,25635.732997,25564.279690,666.427132,1855.088666,25796.285714,122775.857143
Persistence-Enhanced LSTM,1061,1061,25521.587948,25564.279690,1889.050099,1855.088666,25031.448663,124706.254473


,MAE,RMSE,MAPE,sMAPE
Model,,,,
Naive,1290.353242,1853.624774,1.742747,1.744142
Persistence-Enhanced LSTM,1321.365311,1881.091190,1.783956,1.791645
7-Day Moving Average,2209.776153,2999.605073,3.021810,3.024208


## 9. Protocol A vs Protocol B Comparison

The current frozen PE-LSTM vector is Protocol A because each next-day price uses the previous observed close after it becomes available. No frozen recursive Protocol B neural vector exists, so this audit does not invent one or compare aggregate metrics across protocols.


In [10]:
protocol_comparison = pd.DataFrame([
    {"Protocol": "A", "Persistence-Enhanced LSTM vector": "Available and audited", "Corrected Transformer vector": "No frozen artifact", "Ranking eligibility": "PE-LSTM eligible"},
    {"Protocol": "B", "Persistence-Enhanced LSTM vector": "No frozen artifact", "Corrected Transformer vector": "No frozen artifact", "Ranking eligibility": "No neural model eligible"},
]).set_index("Protocol")
protocol_comparison


,Persistence-Enhanced LSTM vector,Corrected Transformer vector,Ranking eligibility
Protocol,,,
A,Available and audited,No frozen artifact,PE-LSTM eligible
B,No frozen artifact,No frozen artifact,No neural model eligible


## 10. Final Diagnosis


In [11]:
diagnosis = pd.DataFrame(
    {
        "Audit Area": ["Implementation validity", "Forecasting-protocol validity", "Model collapse", "Over-smoothing", "Range compression", "Data leakage", "Target misalignment", "Scaling extrapolation"],
        "Evidence to Review": ["Generation notebooks contain model summaries and deterministic checks; this notebook validates saved-vector schema and values.", "Protocol A and Protocol B definitions and artifact availability are reported separately.", "Prediction uniqueness and standard deviation checks.", "Prediction daily-change standard deviation versus actual.", "Prediction range versus actual range.", "Baselines use prior observations only; saved-vector timestamps align exactly.", "Sequence maps confirm a window ending on t-1 predicts t.", "Scaler is fitted on train only; test scaling extrapolation is reported."],
        "Pass Criteria": ["Generation checks pass and the saved artifact passes schema/value validation.", "No cross-protocol ranking is presented as one fair leaderboard.", "Predictions are non-constant except intentional baselines.", "Neural change variance is not silently near zero.", "Compressed ranges are explicitly diagnosed.", "No forecast uses target-day information.", "Every target date is exactly one day after the input window.", "Inverse-scaling roundtrips pass and extrapolation is acknowledged."],
    }
)
diagnosis


,Audit Area,Evidence to Review,Pass Criteria
0,Implementation validity,Generation notebooks contain model summaries a...,Generation checks pass and the saved artifact ...
1,Forecasting-protocol validity,Protocol A and Protocol B definitions and arti...,No cross-protocol ranking is presented as one ...
2,Model collapse,Prediction uniqueness and standard deviation c...,Predictions are non-constant except intentiona...
3,Over-smoothing,Prediction daily-change standard deviation ver...,Neural change variance is not silently near zero.
4,Range compression,Prediction range versus actual range.,Compressed ranges are explicitly diagnosed.
5,Data leakage,Baselines use prior observations only; saved-v...,No forecast uses target-day information.
6,Target misalignment,Sequence maps confirm a window ending on t-1 p...,Every target date is exactly one day after the...
7,Scaling extrapolation,Scaler is fitted on train only; test scaling e...,Inverse-scaling roundtrips pass and extrapolat...


## 11. Audit Execution Guard

This final guard makes the notebook's artifact-only status machine-checkable.


In [12]:
execution_guard = pd.DataFrame({"Check": ["Artifact-only mode", "No model training", "No checkpoint loading", "No forecast regeneration"], "Pass": [ARTIFACT_ONLY, not MODEL_TRAINING_EXECUTED, not MODEL_CHECKPOINT_LOADED, not FORECAST_REGENERATED]})
assert execution_guard["Pass"].all()
execution_guard


,Check,Pass
0,Artifact-only mode,True
1,No model training,True
2,No checkpoint loading,True
3,No forecast regeneration,True
